In [317]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path 
import sys,importlib
import re,ast
import joblib
import json , shutil
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import ndcg_score
from starlette.testclient import TestClient as TestClient

In [2]:
file_path = "data/raw/02_CortexHR_AI_ML_Dataset.xlsx"

In [3]:
resumes= pd.read_excel(file_path,sheet_name="resumes")  
job_descriptions=pd.read_excel(file_path,sheet_name="job_descriptions") 
resume_job_labels=pd.read_excel(file_path,sheet_name="resume_job_labels")

In [4]:
resumes.shape

(185, 15)

In [5]:
job_descriptions.shape

(10, 11)

In [9]:
resume_job_labels.shape

(555, 7)

In [6]:
datasets={
        "Resumes":resumes,
        "job Description": job_descriptions,
        "Resume job labels":resume_job_labels
}

In [7]:
for i, df in  datasets.items():
    print("\n" + "=" * 60)
    print(i)
    print("=" * 60)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nFirst 5 records:")
    print(df.head())


Resumes

Columns:
['resume_id', 'candidate_name', 'email', 'city', 'total_experience_years', 'current_title', 'current_company', 'highest_education', 'college_tier', 'skills', 'certifications', 'source_format', 'layout_difficulty', 'target_job_family', 'resume_text']

First 5 records:
  resume_id     candidate_name                                email  \
0  RES-0001      Girish Joseph      girish.joseph@mail-demo.example   
1  RES-0002    Mythili Abraham    mythili.abraham@mail-demo.example   
2  RES-0003  Sneha Subramanian  sneha.subramanian@mail-demo.example   
3  RES-0004       Arjun Mathew       arjun.mathew@mail-demo.example   
4  RES-0005        Isha Joseph        isha.joseph@mail-demo.example   

         city  total_experience_years              current_title  \
0   Delhi NCR                     4.8                ML Engineer   
1      Mumbai                     3.7               HR Executive   
2       Kochi                     5.7          Software Engineer   
3   Delhi NCR 

In [8]:
for i, df in datasets.items():
    print("\n" + "=" * 60)
    print(i)
    print("=" * 60)
    print("\nMissing values:")
    display(df.isnull().sum())
    print("\nDuplicate rows:", df.duplicated().sum())


Resumes

Missing values:


resume_id                   0
candidate_name              0
email                       0
city                        0
total_experience_years      0
current_title               0
current_company             0
highest_education           0
college_tier                0
skills                      0
certifications            114
source_format               0
layout_difficulty           0
target_job_family           0
resume_text                 0
dtype: int64


Duplicate rows: 0

job Description

Missing values:


job_id                  0
job_title               0
department_id           0
job_level               0
min_experience_years    0
required_skills         0
nice_to_have_skills     0
education_preference    0
work_location           0
openings                0
description_text        0
dtype: int64


Duplicate rows: 0

Resume job labels

Missing values:


resume_id                  0
job_id                     0
human_suitability_score    0
shortlist                  0
required_skill_coverage    0
nice_skill_coverage        0
reviewer_note              0
dtype: int64


Duplicate rows: 0


In [9]:
resumes.isnull().sum()

resume_id                   0
candidate_name              0
email                       0
city                        0
total_experience_years      0
current_title               0
current_company             0
highest_education           0
college_tier                0
skills                      0
certifications            114
source_format               0
layout_difficulty           0
target_job_family           0
resume_text                 0
dtype: int64

In [10]:
# reading 5 smaples resumes 

In [11]:
resumes.head()

,resume_id,candidate_name,email,city,total_experience_years,current_title,current_company,highest_education,college_tier,skills,certifications,source_format,layout_difficulty,target_job_family,resume_text
0,RES-0001,Girish Joseph,girish.joseph@mail-demo.example,Delhi NCR,4.8,ML Engineer,Quantum Leaf Technologies,M.Sc Statistics,Tier 3,MLOps; Communication; TensorFlow; scikit-learn...,NaN,DOCX,Missing sections,J04,GIRISH JOSEPH\nDelhi NCR | girish.joseph@mail-...
1,RES-0002,Mythili Abraham,mythili.abraham@mail-demo.example,Mumbai,3.7,HR Executive,Lakeview Systems,MBA HR,Tier 3,Python; HR Policies; Leadership; Payroll Basic...,NaN,PDF,Clean,J07,MYTHILI ABRAHAM\nMumbai | mythili.abraham@mail...
2,RES-0003,Sneha Subramanian,sneha.subramanian@mail-demo.example,Kochi,5.7,Software Engineer,Bytewave Solutions,M.Tech,Tier 3,Critical Thinking; Python; Machine Learning; T...,NaN,PDF,Clean,J02,SNEHA SUBRAMANIAN\nKochi | sneha.subramanian@m...
3,RES-0004,Arjun Mathew,arjun.mathew@mail-demo.example,Delhi NCR,5.1,Associate Product Manager,Bytewave Solutions,MBA,Tier 3,GST; Critical Thinking; Stakeholder Management...,NaN,DOCX,Clean,J05,ARJUN MATHEW\nDelhi NCR | arjun.mathew@mail-de...
4,RES-0005,Isha Joseph,isha.joseph@mail-demo.example,Coimbatore,1.4,Reporting Analyst,Orion Infotech,B.Com,Tier 2,Critical Thinking; CRM; Statistics; Python; Ti...,Microsoft Power BI Data Analyst,PDF,Clean,J03,ISHA JOSEPH\nCoimbatore | isha.joseph@mail-dem...


In [12]:
resume_sample= resumes[["resume_id","total_experience_years","current_title","skills","target_job_family","resume_text"]].head()

In [13]:
pd.set_option("display.max_colwidth",300)

In [14]:
resume_sample

,resume_id,total_experience_years,current_title,skills,target_job_family,resume_text
0,RES-0001,4.8,ML Engineer,MLOps; Communication; TensorFlow; scikit-learn; PyTorch; Machine Learning; Python; Model Deployment; System Design; SQL; MS Office,J04,"GIRISH JOSEPH\nDelhi NCR | girish.joseph@mail-demo.example\n\nSUMMARY\nExperienced ml engineer with 4.8 years of experience. Strengths include MLOps, Communication, TensorFlow.\n\nEXPERIENCE\nML Engineer, Quantum Leaf Technologies (2.2 yrs)\n- Trained models that improved accuracy by 28%\n- Buil..."
1,RES-0002,3.7,HR Executive,Python; HR Policies; Leadership; Payroll Basics; HRIS; Time Management; Communication; Recruitment,J07,"MYTHILI ABRAHAM\nMumbai | mythili.abraham@mail-demo.example\n\nSUMMARY\nMotivated hr executive with 3.7 years of experience. Strengths include Python, HR Policies, Leadership.\n\nEXPERIENCE\nHR Executive, Lakeview Systems (2.1 yrs)\n- Closed 43 positions in a year\n- Maintained HR records for 61..."
2,RES-0003,5.7,Software Engineer,Critical Thinking; Python; Machine Learning; Teamwork; Product Roadmaps,J02,"SNEHA SUBRAMANIAN\nKochi | sneha.subramanian@mail-demo.example\n\nSUMMARY\nExperienced software engineer with 5.7 years of experience. Strengths include Critical Thinking, Python, Machine Learning.\n\nEXPERIENCE\nSoftware Engineer, Bytewave Solutions (2.9 yrs)\n- Led design of a service handling..."
3,RES-0004,5.1,Associate Product Manager,GST; Critical Thinking; Stakeholder Management; User Research; Time Management; Analytics; Python,J05,"ARJUN MATHEW\nDelhi NCR | arjun.mathew@mail-demo.example\n\nSUMMARY\nExperienced associate product manager with 5.1 years of experience. Strengths include GST, Critical Thinking, Stakeholder Management.\n\nEXPERIENCE\nAssociate Product Manager, Bytewave Solutions (2.8 yrs)\n- Launched features a..."
4,RES-0005,1.4,Reporting Analyst,Critical Thinking; CRM; Statistics; Python; Time Management; Power BI; Data Visualization,J03,"ISHA JOSEPH\nCoimbatore | isha.joseph@mail-demo.example\n\nSUMMARY\nMotivated reporting analyst with 1.4 years of experience. Strengths include Critical Thinking, CRM, Statistics.\n\nEXPERIENCE\nReporting Analyst, Orion Infotech (0.1 yrs)\n- Automated weekly reports and saved 48 hours a month\n-..."


In [15]:
#reading 3 job_description

In [16]:
job_descriptions.head()

,job_id,job_title,department_id,job_level,min_experience_years,required_skills,nice_to_have_skills,education_preference,work_location,openings,description_text
0,J01,Software Engineer,D01,2,2,Python; SQL; REST APIs; Git; Data Structures,Docker; AWS; Django; FastAPI; PostgreSQL; Unit Testing,B.E./B.Tech in Computer Science or related,Bengaluru / Chennai / Hyderabad (hybrid),4,"Zenith Nova Technologies is hiring a Software Engineer for the D01 team. You will own day-to-day delivery in your area, work closely with colleagues across departments and help us improve how we serve customers. Must have: Python, SQL, REST APIs, Git, Data Structures. Good to have: Docker, AWS, ..."
1,J02,Senior Software Engineer,D01,3,6,Python; System Design; REST APIs; PostgreSQL; Docker,Kubernetes; AWS; Microservices; CI/CD; Mentoring,B.E./B.Tech or MCA,Bengaluru / Chennai / Hyderabad (hybrid),3,"Zenith Nova Technologies is hiring a Senior Software Engineer for the D01 team. You will own day-to-day delivery in your area, work closely with colleagues across departments and help us improve how we serve customers. Must have: Python, System Design, REST APIs, PostgreSQL, Docker. Good to have..."
2,J03,Data Analyst,D03,2,2,SQL; Excel; Python; Data Visualization; Statistics,Power BI; Tableau; pandas; A/B Testing; Apache Superset,Bachelor's in a quantitative field,Bengaluru / Chennai / Hyderabad (hybrid),4,"Zenith Nova Technologies is hiring a Data Analyst for the D03 team. You will own day-to-day delivery in your area, work closely with colleagues across departments and help us improve how we serve customers. Must have: SQL, Excel, Python, Data Visualization, Statistics. Good to have: Power BI, Ta..."
3,J04,ML Engineer,D03,3,3,Python; Machine Learning; scikit-learn; SQL; Model Deployment,TensorFlow; PyTorch; MLOps; Docker; NLP,"B.Tech/M.Tech/M.Sc in CS, Statistics or related",Bengaluru / Chennai / Hyderabad (hybrid),4,"Zenith Nova Technologies is hiring a ML Engineer for the D03 team. You will own day-to-day delivery in your area, work closely with colleagues across departments and help us improve how we serve customers. Must have: Python, Machine Learning, scikit-learn, SQL, Model Deployment. Good to have: Te..."
4,J05,Product Manager,D02,3,5,Product Roadmaps; User Research; Stakeholder Management; Agile; Analytics,Figma; SQL; A/B Testing; Go-to-Market,Bachelor's degree; MBA preferred,Bengaluru / Chennai / Hyderabad (hybrid),4,"Zenith Nova Technologies is hiring a Product Manager for the D02 team. You will own day-to-day delivery in your area, work closely with colleagues across departments and help us improve how we serve customers. Must have: Product Roadmaps, User Research, Stakeholder Management, Agile, Analytics. ..."


In [17]:
job_sample=job_descriptions[["job_id","job_title","job_level","min_experience_years","required_skills",
                             "nice_to_have_skills","description_text"]].head(3)

In [18]:
job_sample

,job_id,job_title,job_level,min_experience_years,required_skills,nice_to_have_skills,description_text
0,J01,Software Engineer,2,2,Python; SQL; REST APIs; Git; Data Structures,Docker; AWS; Django; FastAPI; PostgreSQL; Unit Testing,"Zenith Nova Technologies is hiring a Software Engineer for the D01 team. You will own day-to-day delivery in your area, work closely with colleagues across departments and help us improve how we serve customers. Must have: Python, SQL, REST APIs, Git, Data Structures. Good to have: Docker, AWS, ..."
1,J02,Senior Software Engineer,3,6,Python; System Design; REST APIs; PostgreSQL; Docker,Kubernetes; AWS; Microservices; CI/CD; Mentoring,"Zenith Nova Technologies is hiring a Senior Software Engineer for the D01 team. You will own day-to-day delivery in your area, work closely with colleagues across departments and help us improve how we serve customers. Must have: Python, System Design, REST APIs, PostgreSQL, Docker. Good to have..."
2,J03,Data Analyst,2,2,SQL; Excel; Python; Data Visualization; Statistics,Power BI; Tableau; pandas; A/B Testing; Apache Superset,"Zenith Nova Technologies is hiring a Data Analyst for the D03 team. You will own day-to-day delivery in your area, work closely with colleagues across departments and help us improve how we serve customers. Must have: SQL, Excel, Python, Data Visualization, Statistics. Good to have: Power BI, Ta..."


In [19]:
## Manual Resume and Job Description Analysis

'''Five resumes and three job descriptions were inspected.

### Key observations
1. Required skills are the most important indicator of job relevance.
2. Nice-to-have skills provide additional evidence of suitability.
3. Total experience must be compared against the minimum job requirement.
4. Resume work descriptions help distinguish practical experience from simple keyword mentions.
5. Different job families require different skill sets.

### Fairness considerations
The ranking system will exclude candidate names, email addresses, gender, age, location and college tier from scoring.

### Initial conclusion
A combination of TF-IDF similarity, required skill overlap, preferred skill overlap and experience relevance should provide a more explainable ranking than using experience alone.
'''

'Five resumes and three job descriptions were inspected.\n\n### Key observations\n1. Required skills are the most important indicator of job relevance.\n2. Nice-to-have skills provide additional evidence of suitability.\n3. Total experience must be compared against the minimum job requirement.\n4. Resume work descriptions help distinguish practical experience from simple keyword mentions.\n5. Different job families require different skill sets.\n\n### Fairness considerations\nThe ranking system will exclude candidate names, email addresses, gender, age, location and college tier from scoring.\n\n### Initial conclusion\nA combination of TF-IDF similarity, required skill overlap, preferred skill overlap and experience relevance should provide a more explainable ranking than using experience alone.\n'

In [20]:
required_skill_set=set()
preferred_skill_set=set()

In [21]:
for skills in job_descriptions["required_skills"].dropna():
    required_skill_set.update(
        skill.strip().lower()
        for skill in skills.split(";")
    )

In [22]:
for skills in job_descriptions["nice_to_have_skills"].dropna():
    preferred_skill_set.update(
        skill.strip().lower()
        for skill in skills.split(";")
    )

In [23]:
skills_vocabulary=sorted(required_skill_set.union(preferred_skill_set))                     

In [24]:
len(required_skill_set)

43

In [25]:
len(preferred_skill_set)

41

In [26]:
len(skills_vocabulary)

80

In [27]:
skills_vocabulary

['a/b testing',
 'account planning',
 'agile',
 'analytics',
 'apache superset',
 'audit support',
 'aws',
 'b2b sales',
 'bank reconciliation',
 'bookkeeping',
 'ci/cd',
 'cold outreach',
 'communication',
 'content marketing',
 'copywriting',
 'crm',
 'customer communication',
 'data structures',
 'data visualization',
 'django',
 'docker',
 'email campaigns',
 'empathy',
 'employee onboarding',
 'engagement surveys',
 'english proficiency',
 'escalation handling',
 'excel',
 'fastapi',
 'figma',
 'financial reporting',
 'git',
 'go-to-market',
 'google ads',
 'google analytics',
 'gst',
 'hr policies',
 'hris',
 'hubspot',
 'kubernetes',
 'labour law basics',
 'lead generation',
 'machine learning',
 'mentoring',
 'microservices',
 'mlops',
 'model deployment',
 'ms excel',
 'negotiation',
 'nlp',
 'pandas',
 'payroll basics',
 'postgresql',
 'power bi',
 'presentation skills',
 'problem solving',
 'product roadmaps',
 'python',
 'pytorch',
 'recruitment',
 'regional language fluenc

In [28]:
required_skill_set

{'agile',
 'analytics',
 'b2b sales',
 'bank reconciliation',
 'bookkeeping',
 'communication',
 'content marketing',
 'crm',
 'customer communication',
 'data structures',
 'data visualization',
 'docker',
 'email campaigns',
 'empathy',
 'employee onboarding',
 'english proficiency',
 'excel',
 'git',
 'google analytics',
 'gst',
 'hr policies',
 'lead generation',
 'machine learning',
 'model deployment',
 'ms excel',
 'negotiation',
 'postgresql',
 'presentation skills',
 'problem solving',
 'product roadmaps',
 'python',
 'recruitment',
 'rest apis',
 'scikit-learn',
 'seo',
 'social media marketing',
 'sql',
 'stakeholder management',
 'statistics',
 'system design',
 'tally',
 'ticketing tools',
 'user research'}

In [29]:
preferred_skill_set

{'a/b testing',
 'account planning',
 'apache superset',
 'audit support',
 'aws',
 'ci/cd',
 'cold outreach',
 'copywriting',
 'crm',
 'django',
 'docker',
 'engagement surveys',
 'escalation handling',
 'fastapi',
 'figma',
 'financial reporting',
 'go-to-market',
 'google ads',
 'hris',
 'hubspot',
 'kubernetes',
 'labour law basics',
 'mentoring',
 'microservices',
 'mlops',
 'nlp',
 'pandas',
 'payroll basics',
 'postgresql',
 'power bi',
 'pytorch',
 'regional language fluency',
 'saas sales',
 'salesforce',
 'sap',
 'sql',
 'tableau',
 'tds',
 'tensorflow',
 'unit testing',
 'zendesk'}

In [30]:
#Extract skills from resume text 

In [31]:
def extract_skills(resume_text, vocabulary):
    text = str(resume_text).lower()
    extracted = []
    for skill in vocabulary:
        pattern = r"(?<!\w)" + re.escape(skill) + r"(?!\w)"
        if re.search(pattern, text):
            extracted.append(skill)
    return sorted(set(extracted))
resumes["extracted_skills"] = resumes["resume_text"].apply(
    lambda text: extract_skills(text, skills_vocabulary))
display(resumes[["resume_id", "extracted_skills"]].head(10))

,resume_id,extracted_skills
0,RES-0001,"[communication, machine learning, mlops, model deployment, python, pytorch, scikit-learn, sql, system design, tensorflow]"
1,RES-0002,"[communication, hr policies, hris, payroll basics, python, recruitment]"
2,RES-0003,"[machine learning, product roadmaps, python]"
3,RES-0004,"[analytics, gst, python, stakeholder management, user research]"
4,RES-0005,"[crm, data visualization, power bi, python, statistics]"
5,RES-0006,"[communication, empathy, escalation handling, recruitment, sql, ticketing tools]"
6,RES-0007,"[account planning, b2b sales, cold outreach, crm, lead generation, negotiation, presentation skills, python, recruitment, saas sales]"
7,RES-0008,"[docker, machine learning, python, scikit-learn, sql, tensorflow]"
8,RES-0009,"[employee onboarding, mlops, model deployment, nlp, python, pytorch, seo, sql, tensorflow]"
9,RES-0010,"[bank reconciliation, communication, employee onboarding, excel, financial reporting, gst, ms excel, python, sap, tally]"


In [ ]:
# Extract total years of experience from resume text

In [33]:
def extract_experience(resume_text, fallback):
    text = str(resume_text).lower()
    match = re.search(
        r"(\d+(?:\.\d+)?)\s+years?\s+of\s+experience",text )
    if match:
        return float(match.group(1))
    return float(fallback)
resumes["extracted_experience_years"] = resumes.apply(
    lambda row: extract_experience(
        row["resume_text"],
        row["total_experience_years"]),axis=1)
display(resumes[["resume_id","extracted_experience_years","extracted_skills"]].head(10))

,resume_id,extracted_experience_years,extracted_skills
0,RES-0001,4.8,"[communication, machine learning, mlops, model deployment, python, pytorch, scikit-learn, sql, system design, tensorflow]"
1,RES-0002,3.7,"[communication, hr policies, hris, payroll basics, python, recruitment]"
2,RES-0003,5.7,"[machine learning, product roadmaps, python]"
3,RES-0004,5.1,"[analytics, gst, python, stakeholder management, user research]"
4,RES-0005,1.4,"[crm, data visualization, power bi, python, statistics]"
5,RES-0006,2.9,"[communication, empathy, escalation handling, recruitment, sql, ticketing tools]"
6,RES-0007,2.8,"[account planning, b2b sales, cold outreach, crm, lead generation, negotiation, presentation skills, python, recruitment, saas sales]"
7,RES-0008,4.3,"[docker, machine learning, python, scikit-learn, sql, tensorflow]"
8,RES-0009,3.8,"[employee onboarding, mlops, model deployment, nlp, python, pytorch, seo, sql, tensorflow]"
9,RES-0010,3.4,"[bank reconciliation, communication, employee onboarding, excel, financial reporting, gst, ms excel, python, sap, tally]"


In [34]:
experience_comparison = pd.DataFrame({
    "resume_id": resumes["resume_id"],
    "original_experience": resumes["total_experience_years"],
    "extracted_experience": resumes["extracted_experience_years"]
})
experience_comparison["matches"] = np.isclose(
    experience_comparison["original_experience"],
    experience_comparison["extracted_experience"]
)
print("Total resumes:", len(resumes))
print("Matching experience values:", experience_comparison["matches"].sum())
print("Mismatches:", (~experience_comparison["matches"]).sum())
display(experience_comparison[~experience_comparison["matches"]].head(10))

Total resumes: 185
Matching experience values: 185
Mismatches: 0


,resume_id,original_experience,extracted_experience,matches


In [37]:
print("Total resumes:", len(resumes))
print("Average extracted skills:", round(resumes["skill_count"].mean(), 2))
print("Minimum extracted skills:", resumes["skill_count"].min())
print("Maximum extracted skills:", resumes["skill_count"].max())
display(resumes[["resume_id", "skill_count", "extracted_experience_years"]].head(10))

Total resumes: 185
Average extracted skills: 7.36
Minimum extracted skills: 1
Maximum extracted skills: 13


,resume_id,skill_count,extracted_experience_years
0,RES-0001,10,4.8
1,RES-0002,6,3.7
2,RES-0003,3,5.7
3,RES-0004,5,5.1
4,RES-0005,5,1.4
5,RES-0006,6,2.9
6,RES-0007,10,2.8
7,RES-0008,6,4.3
8,RES-0009,9,3.8
9,RES-0010,10,3.4


In [38]:
# ranking engine 

In [39]:
 len(resume_job_labels)

555

In [40]:
resume_job_labels["human_suitability_score"].describe()

count    555.000000
mean       2.336937
std        1.056177
min        1.000000
25%        2.000000
50%        2.000000
75%        3.000000
max        5.000000
Name: human_suitability_score, dtype: float64

In [41]:
display(resume_job_labels["shortlist"].value_counts(dropna=False))

shortlist
N    460
Y     95
Name: count, dtype: int64

In [42]:
resume_job_labels.head(10)

,resume_id,job_id,human_suitability_score,shortlist,required_skill_coverage,nice_skill_coverage,reviewer_note
0,RES-0001,J04,4,Y,1.0,0.6,Strong match
1,RES-0001,J09,2,N,0.0,0.0,Weak match
2,RES-0001,J03,2,N,0.4,0.0,Weak match
3,RES-0002,J07,3,N,0.6,0.5,Partial match
4,RES-0002,J05,2,N,0.0,0.0,Weak match
5,RES-0002,J02,2,N,0.2,0.0,Weak match
6,RES-0003,J02,2,N,0.2,0.0,Weak match
7,RES-0003,J03,2,N,0.2,0.0,Weak match
8,RES-0003,J01,2,N,0.2,0.0,Weak match
9,RES-0004,J05,3,N,0.6,0.0,Partial match


In [45]:
resume_job_labels["resume_id"].nunique()

185

In [46]:
 resume_job_labels["job_id"].nunique()

10

In [47]:
display(
    resume_job_labels.groupby("job_id").agg(
        total_candidates=("resume_id", "count"),
        shortlisted=("shortlist", "sum"),
        average_human_score=("human_suitability_score", "mean")
    )
)

,total_candidates,shortlisted,average_human_score
job_id,,,
J01,64,NNNNNNNNNNNNYNNNNNNNNNNNYNNNNNNNNNNNYNNNNYNNNNNNYNYNNNNYNNNNNNNN,2.281250
J02,47,NNNNNNNNNNYNNNNNNNNNNNYNYNNNNNNNNNNNNNNNNNNNNNN,1.957447
J03,60,NNNNNYNNNNNNNNNNNNNNNNNNNNNYNNNYYNNNNNNNNYNYNYYYNNNNNNYNYNYN,2.516667
J04,62,YNYNNNNNNYYYNNNNNNNNYNNNNNYNNYNYNNNNNNNNNNNNNNYNYYYNNNNNNNYYYN,2.580645
J05,57,NNNNYNYNYNNNNYNNNNNYNNNNNNYNNNNNNNNNNYNNNNNNNNNNNNNYNNYNN,2.210526
J06,49,NNNYYNNNYYNNNNNNNNNNNNNNNNNNNYNNNNNNNNNYNNNNNNNNN,2.224490
J07,49,NNNNYNNNNNNYNYNNYNYYNYNNNNNNNNNNNNNNNNNNNNNNYNNNN,2.408163
J08,55,NNNYNNYNNNNNNNNNYYNYNNNNNNNNNNNNNYYNNNNNNNNNNNNNYYNNNNN,2.363636
J09,57,NNNYYNNYNNNNYNNNNYNYNNYNNNNNNNNNNNNNNYYNNNYYNNYNNNNYNNYNN,2.403509


In [51]:
human_summary = (
    resume_job_labels
    .assign(
        shortlisted_flag=lambda df:
            df["shortlist"].astype(str).str.upper().eq("Y")
    )
    .groupby("job_id")
    .agg(
        total_candidates=("resume_id", "count"),
        shortlisted=("shortlisted_flag", "sum"),
        average_human_score=("human_suitability_score", "mean")
    )
    .reset_index()
)
human_summary["shortlist_rate"] = (
    human_summary["shortlisted"] /
    human_summary["total_candidates"] * 100
).round(2)
display(human_summary)

,job_id,total_candidates,shortlisted,average_human_score,shortlist_rate
0,J01,64,7,2.281250,10.94
1,J02,47,3,1.957447,6.38
2,J03,60,12,2.516667,20.00
3,J04,62,16,2.580645,25.81
4,J05,57,9,2.210526,15.79
5,J06,49,6,2.224490,12.24
6,J07,49,8,2.408163,16.33
7,J08,55,9,2.363636,16.36
8,J09,57,14,2.403509,24.56
9,J10,55,11,2.327273,20.00


In [52]:
# remove personal identifiers from resume text 

In [136]:
def clean_resume(text, candidate_name=None, city=None):
    text = str(text)
    if candidate_name and pd.notna(candidate_name):
        text = re.sub(
            re.escape(str(candidate_name)),
            " ",
            text,
            flags=re.IGNORECASE
        )
    text = re.sub(
        r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b",
        " ",
        text,
        flags=re.IGNORECASE
    )
    clean_lines = []
    for line in text.splitlines():
        if "@" in line:
            continue
        if city and pd.notna(city):
            line = re.sub(
                re.escape(str(city)),
                " ",
                line,
                flags=re.IGNORECASE
            )
        clean_lines.append(line)
    text = " ".join(clean_lines)
    text = re.sub(
        r"\b(?:age|gender|sex)\s*[:=]\s*[\w-]+",
        " ",
        text,
        flags=re.IGNORECASE
    )
    text = re.sub(
        r"\b(?:college\s*)?tier\s*[1-5](?:\s+(?:institution|college))?\b",
        " ",
        text,
        flags=re.IGNORECASE
    )
    return re.sub(r"\s+", " ", text).strip()


In [140]:
resumes["clean_resume_text"] = resumes.apply(lambda row: sanitize_resume(row["resume_text"],row["candidate_name"],row["city"]),axis=1)
tier_remaining = resumes["clean_resume_text"].str.contains(r"\btier\s*[1-5]\b",case=False,regex=True,na=False).sum()
age_gender_remaining = resumes["clean_resume_text"].str.contains(
    r"\b(?:age|gender|sex)\s*[:=]",case=False,regex=True,na=False).sum()
tier_remaining

np.int64(0)

In [139]:
age_gender_remaining

np.int64(0)

In [141]:
display(resumes[resumes["resume_id"].isin(["RES-0129", "RES-0119", "RES-0077", "RES-0012", "RES-0088"] ) ][["resume_id", "clean_resume_text"]])

,resume_id,clean_resume_text
11,RES-0012,"| SUMMARY Motivated business development associate with 3.7 years of experience. Strengths include Account Planning, B2B Sales, Python. EXPERIENCE Business Development Associate, Silverline Digital (1.7 yrs) - Generated 26 qualified leads every month - Generated 36 qualified leads every month Bu..."
76,RES-0077,"| SUMMARY Experienced ml engineer with 5.7 years of experience. Strengths include Time Management, Recruitment, Teamwork. EXPERIENCE ML Engineer, Veda Networks (3.8 yrs) - Deployed a model serving 79k predictions a day - Trained models that improved accuracy by 33% Software Engineer, Pinnacle Ou..."
87,RES-0088,"EXPERIENCE | Data Analyst, Nimbus Retail Tech (0 yrs) - Deployed a model serving 6k predictions a day SUMMARY - Built feature pipelines for 10 models Motivated data analyst with 2.8 years of experience. Strengths include scikit-learn, Teamwork, TensorFlow.Data Analyst, Orion Infotech (2.8 yrs) -..."
118,RES-0119,| SUMMARY Experienced software engineer with 5.9 years of experience. Strengths include Teamwork | Leadership | Docker. EXPERIENCE Software Engineer | Harbor Analytics (4.4 yrs) - Cut deployment time by 54% by introducing CI/CD - Cut deployment time by 15% by introducing CI/CD Software Engineer ...
128,RES-0129,"| SUMMARY Experienced full-stack developer with 6.1 years of experience. Strengths include Ticketing Tools, Kubernetes, PostgreSQL. EXPERIENCE Full-Stack Developer, Mapleton Services (3.3 yrs) - Mentored 54 junior engineers - Mentored 89 junior engineers Full-Stack Developer, Nimbus Retail Tech ..."


In [142]:
#TF-IDF

In [202]:
resumes = resumes.reset_index(drop=True)
job_descriptions = job_descriptions.reset_index(drop=True)

In [203]:
corpus=(resumes["clean_resume_text"].fillna("").tolist()+ job_descriptions["description_text"].fillna("").tolist())

In [204]:
tfidf_vectorizer=TfidfVectorizer(stop_words="english",ngram_range=(1, 2),min_df=1)

In [205]:
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

In [206]:
resume_vectors = tfidf_matrix[:len(resumes)]
job_vectors = tfidf_matrix[len(resumes):]

In [207]:
similarity_matrix = cosine_similarity(resume_vectors,job_vectors)

In [208]:
resume_vectors.shape

(185, 3205)

In [209]:
job_vectors.shape

(10, 3205)

In [210]:
 similarity_matrix.shape

(185, 10)

In [214]:
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=resumes["resume_id"],
    columns=job_descriptions["job_id"])

In [215]:
first_resume_matches = (
    similarity_df.loc["RES-0001"]
    .sort_values(ascending=False)
    .rename("tfidf_similarity")
    .reset_index()
)

In [216]:
first_resume_matches

,job_id,tfidf_similarity
0,J04,0.143214
1,J03,0.025067
2,J02,0.020950
3,J01,0.020322
4,J07,0.009993
5,J06,0.008288
6,J10,0.008014
7,J05,0.006827
8,J08,0.006562
9,J09,0.005235


In [217]:
def parse_skills(skill_text):
    if pd.isna(skill_text):
        return set()
    return {
        skill.strip().lower()
        for skill in str(skill_text).split(";")
        if skill.strip()
    }
ranking_rows = []
for resume_idx, resume in resumes.iterrows():
    resume_skills = set(resume["extracted_skills"])
    candidate_exp = float(resume["extracted_experience_years"])
    for job_idx, job in job_descriptions.iterrows():
        required_skills = parse_skills(job["required_skills"])
        preferred_skills = parse_skills(job["nice_to_have_skills"])
        required_matches = resume_skills.intersection(required_skills)
        required_coverage = (
            len(required_matches) / len(required_skills)
            if required_skills else 0
        )
        preferred_matches = resume_skills.intersection(preferred_skills)

        preferred_coverage = (
            len(preferred_matches) / len(preferred_skills)
            if preferred_skills else 0
        )
        minimum_exp = float(job["min_experience_years"])
        experience_score = (
            min(candidate_exp / minimum_exp, 1.0)
            if minimum_exp > 0 else 1.0
        )
        tfidf_score = float(similarity_matrix[resume_idx, job_idx])
        final_score = 100 * (
            0.40 * tfidf_score
            + 0.35 * required_coverage
            + 0.15 * preferred_coverage
            + 0.10 * experience_score
        )
        ranking_rows.append({
            "resume_id": resume["resume_id"],
            "job_id": job["job_id"],
            "tfidf_similarity": tfidf_score,
            "required_skill_coverage": required_coverage,
            "preferred_skill_coverage": preferred_coverage,
            "experience_relevance": experience_score,
            "ranking_score": final_score
        })
ranking_results = pd.DataFrame(ranking_rows)
print("Total resume-job rankings:", len(ranking_results))
print("Unique resumes:", ranking_results["resume_id"].nunique())
print("Unique jobs:", ranking_results["job_id"].nunique())
display(ranking_results.head(10))

Total resume-job rankings: 1850
Unique resumes: 185
Unique jobs: 10


,resume_id,job_id,tfidf_similarity,required_skill_coverage,preferred_skill_coverage,experience_relevance,ranking_score
0,RES-0001,J01,0.020322,0.4,0.00,1.00,24.812879
1,RES-0001,J02,0.020950,0.4,0.00,0.80,22.837997
2,RES-0001,J03,0.025067,0.4,0.00,1.00,25.002689
3,RES-0001,J04,0.143214,1.0,0.60,1.00,59.728551
4,RES-0001,J05,0.006827,0.0,0.25,0.96,13.623075
5,RES-0001,J06,0.008288,0.0,0.00,1.00,10.331525
6,RES-0001,J07,0.009993,0.2,0.00,1.00,17.399715
7,RES-0001,J08,0.006562,0.0,0.00,1.00,10.262492
8,RES-0001,J09,0.005235,0.0,0.00,1.00,10.209385
9,RES-0001,J10,0.008014,0.0,0.00,1.00,10.320565


In [156]:
#Rank the candidates separately for each job 

In [218]:
ranked_candidates= ranking_results.sort_values(["job_id","ranking_score"],ascending=[True,False]).copy()

In [219]:
ranked_candidates["model_rank"] = (ranked_candidates.groupby("job_id").cumcount() + 1)

In [220]:
len(ranked_candidates)

1850

In [221]:
display(ranked_candidates.groupby("job_id").size() .reset_index(name="candidate_count"))

,job_id,candidate_count
0,J01,185
1,J02,185
2,J03,185
3,J04,185
4,J05,185
5,J06,185
6,J07,185
7,J08,185
8,J09,185
9,J10,185


In [222]:
j04_top10=( ranked_candidates[ranked_candidates["job_id"] == "J04"].head(10))

In [260]:
j04_top10[
    [
        "model_rank",
        "resume_id",
        "ranking_score",
        "tfidf_similarity",
        "required_skill_coverage",
        "preferred_skill_coverage",
        "experience_relevance"
    ]]

,model_rank,resume_id,ranking_score,tfidf_similarity,required_skill_coverage,preferred_skill_coverage,experience_relevance
1393,1,RES-0140,63.089478,0.152237,1.0,0.8,1.000000
303,2,RES-0031,60.469172,0.161729,1.0,0.6,1.000000
653,3,RES-0066,59.858231,0.146456,1.0,0.6,1.000000
1773,4,RES-0178,59.858231,0.146456,1.0,0.6,1.000000
3,5,RES-0001,59.728551,0.143214,1.0,0.6,1.000000
903,6,RES-0091,57.543552,0.163589,1.0,0.4,1.000000
873,7,RES-0088,54.486866,0.103838,1.0,0.4,0.933333
73,8,RES-0008,50.662308,0.166558,0.8,0.4,1.000000
293,9,RES-0030,50.322174,0.133054,1.0,0.0,1.000000
1323,10,RES-0133,49.479440,0.136986,0.8,0.4,1.000000


In [261]:
'''
display(
    job_descriptions.loc[
        job_descriptions["job_id"] == "J02",
        [
            "required_skills",
            "nice_to_have_skills",
            "description_text"
        ]
    ]
)
'''

,required_skills,nice_to_have_skills,description_text
1,Python; System Design; REST APIs; PostgreSQL; Docker,Kubernetes; AWS; Microservices; CI/CD; Mentoring,"Zenith Nova Technologies is hiring a Senior Software Engineer for the D01 team. You will own day-to-day delivery in your area, work closely with colleagues across departments and help us improve how we serve customers. Must have: Python, System Design, REST APIs, PostgreSQL, Docker. Good to have..."


In [262]:
'''
j02 = job_descriptions.loc[
    job_descriptions["job_id"] == "J02"
].iloc[0]

print("FULL JOB DESCRIPTION:")
print(j02["description_text"])

print("\nPREFERRED SKILL CHECK:")

for skill in parse_skills(j02["nice_to_have_skills"]):
    present = bool(
        re.search(
            r"(?<!\w)" + re.escape(skill) + r"(?!\w)",
            str(j02["description_text"]),
            flags=re.IGNORECASE
        )
    )

    print(f"{skill}: {'Present' if present else 'Missing'}")
'''

FULL JOB DESCRIPTION:
Zenith Nova Technologies is hiring a Senior Software Engineer for the D01 team. You will own day-to-day delivery in your area, work closely with colleagues across departments and help us improve how we serve customers. Must have: Python, System Design, REST APIs, PostgreSQL, Docker. Good to have: Kubernetes, AWS, Microservices, CI/CD. Minimum 6 year(s) of relevant experience. Education: B.E./B.Tech or MCA. Location: Bengaluru, Chennai or Hyderabad (hybrid).

PREFERRED SKILL CHECK:
ci/cd: Present
aws: Present
mentoring: Missing
kubernetes: Present
microservices: Present


In [226]:
# comparing the model with human ranking 

In [227]:
human_labels=resume_job_labels[ [
        "resume_id",
        "job_id",
        "human_suitability_score",
        "shortlist",
        "reviewer_note"
    ]
].copy()

In [228]:
evaluation = ranked_candidates.merge(human_labels,on=["resume_id", "job_id"],how="inner")

In [229]:
evaluation["shortlist_flag"] = (evaluation["shortlist"].astype(str).str.strip().str.upper().eq("Y"))

In [230]:
len(evaluation)

555

In [231]:
evaluation["job_id"].nunique()

10

In [232]:
#quality metrics

In [233]:
quality_rows = []

In [234]:
for job_id, group in evaluation.groupby("job_id"):
    model_order = group.sort_values(
        "ranking_score",
        ascending=False
    )
    model_top5 = model_order.head(5)
    human_order = group.sort_values(
        "human_suitability_score",
        ascending=False
    )
    human_top5 = human_order.head(5)
    top5_hits = int(model_top5["shortlist_flag"].sum())
    total_shortlisted = int(group["shortlist_flag"].sum())
    precision_at_5 = top5_hits / len(model_top5)
    recall_at_5 = (
        top5_hits / total_shortlisted
        if total_shortlisted > 0 else 0
    )
    ndcg_at_5 = ndcg_score(
        [group["human_suitability_score"].to_numpy()],
        [group["ranking_score"].to_numpy()],
        k=5
    )
    top5_overlap = len(
        set(model_top5["resume_id"])
        .intersection(set(human_top5["resume_id"]))
    )
    quality_rows.append({
        "job_id": job_id,
        "reviewed_candidates": len(group),
        "human_shortlisted": total_shortlisted,
        "model_top5_hits": top5_hits,
        "precision_at_5_pct": round(precision_at_5 * 100, 2),
        "recall_at_5_pct": round(recall_at_5 * 100, 2),
        "ndcg_at_5": round(float(ndcg_at_5), 4),
        "top5_overlap": top5_overlap
    })

In [235]:
ranking_quality = pd.DataFrame(quality_rows)

In [236]:
ranking_quality.sort_values("job_id")

,job_id,reviewed_candidates,human_shortlisted,model_top5_hits,precision_at_5_pct,recall_at_5_pct,ndcg_at_5,top5_overlap
0,J01,64,7,5,100.0,71.43,1.0000,5
1,J02,47,3,2,40.0,66.67,0.9544,4
2,J03,60,12,5,100.0,41.67,1.0000,5
3,J04,62,16,5,100.0,31.25,0.9352,3
4,J05,57,9,5,100.0,55.56,1.0000,5
5,J06,49,6,5,100.0,83.33,0.9950,5
6,J07,49,8,5,100.0,62.50,1.0000,5
7,J08,55,9,5,100.0,55.56,1.0000,5
8,J09,57,14,5,100.0,35.71,0.9572,4
9,J10,55,11,5,100.0,45.45,0.9969,5


In [237]:
#identifying five misranked candidates

In [238]:
error_analysis=evaluation.copy()

In [239]:
error_analysis["model_e_rank"]=(error_analysis.groupby("job_id")["ranking_score"].rank(method="first", ascending=False).astype(int))

In [240]:
error_analysis["human_rank"] = (error_analysis.groupby("job_id")["human_suitability_score"].rank(method="min", ascending=False).astype(int))

In [241]:
false_positives = error_analysis[(error_analysis["model_e_rank"] <= 5)& (~error_analysis["shortlist_flag"])].copy()
false_negatives = error_analysis[(error_analysis["model_e_rank"] > 5)& (error_analysis["shortlist_flag"])].copy()
print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

False positives: 3
False negatives: 48


In [242]:
selected_fp = false_positives.sort_values( "ranking_score",ascending=False).head(3)

In [243]:
selected_fn = false_negatives.sort_values(["human_suitability_score", "ranking_score"],ascending=[False, True]).head(2)

In [244]:
misranked=pd.concat([selected_fp,selected_fn],ignore_index=True)

In [245]:
misranked["error_type"] = (["False Positive"] * len(selected_fp)+ ["False Negative"] * len(selected_fn))

In [246]:
 misranked[
        [
            "resume_id",
            "job_id",
            "error_type",
            "model_e_rank",
            "human_rank",
            "ranking_score",
            "human_suitability_score",
            "shortlist",
            "tfidf_similarity",
            "required_skill_coverage",
            "preferred_skill_coverage",
            "experience_relevance",
            "reviewer_note"
        ]
    ]

,resume_id,job_id,error_type,model_e_rank,human_rank,ranking_score,human_suitability_score,shortlist,tfidf_similarity,required_skill_coverage,preferred_skill_coverage,experience_relevance,reviewer_note
0,RES-0129,J02,False Positive,3,4,46.829161,3,N,0.095729,0.6,0.80,1.000000,Partial match
1,RES-0119,J02,False Positive,4,4,40.894570,3,N,0.176531,0.6,0.20,0.983333,Partial match
2,RES-0077,J02,False Positive,5,4,25.806217,3,N,0.057655,0.4,0.00,0.950000,Partial match
3,RES-0012,J09,False Negative,19,1,29.957975,5,Y,0.055199,0.4,0.25,1.000000,Strong match
4,RES-0088,J04,False Negative,7,1,54.486866,5,Y,0.103838,1.0,0.40,0.933333,Strong match


In [268]:
evidence_dir = Path("reports/task11")
evidence_dir.mkdir(parents=True, exist_ok=True)
ranking_quality.to_csv(evidence_dir / "ranking_quality.csv",index=False)
misranked.to_csv(evidence_dir / "misranked_candidates.csv",index=False)
ranked_candidates.to_csv(evidence_dir / "all_resume_job_rankings.csv",index=False)

In [269]:
evidence_dir

PosixPath('reports/task11')

In [249]:
candidate_details = misranked.merge(resumes[ [ "resume_id", "current_title", "total_experience_years","extracted_experience_years","extracted_skills","clean_resume_text" ]],on="resume_id",how="left")

In [250]:
for _, row in candidate_details.iterrows():
    print("\n" + "=" * 75)
    print("RESUME:", row["resume_id"])
    print("JOB:", row["job_id"])
    print("ERROR TYPE:", row["error_type"])
    print("\nCurrent title:", row["current_title"])
    print("Experience:", row["extracted_experience_years"], "years")
    print("\nExtracted skills:")
    print(row["extracted_skills"])
    print("\nClean resume text:")
    print(row["clean_resume_text"])


RESUME: RES-0129
JOB: J02
ERROR TYPE: False Positive

Current title: Full-Stack Developer
Experience: 6.1 years

Extracted skills:
['aws', 'ci/cd', 'docker', 'kubernetes', 'microservices', 'postgresql', 'python', 'seo', 'ticketing tools']

Clean resume text:
| SUMMARY Experienced full-stack developer with 6.1 years of experience. Strengths include Ticketing Tools, Kubernetes, PostgreSQL. EXPERIENCE Full-Stack Developer, Mapleton Services (3.3 yrs) - Mentored 54 junior engineers - Mentored 89 junior engineers Full-Stack Developer, Nimbus Retail Tech (2.8 yrs) - Cut deployment time by 81% by introducing CI/CD SKILLS Ticketing Tools, Kubernetes, PostgreSQL, SEO, Docker, Microservices, MS Office, Python, Teamwork EDUCATION B.E. Computer Science, CERTIFICATIONS AWS Solutions Architect Associate

RESUME: RES-0119
JOB: J02
ERROR TYPE: False Positive

Current title: Software Engineer
Experience: 5.9 years

Extracted skills:
['analytics', 'ci/cd', 'crm', 'docker', 'python', 'rest apis']

Clean

In [251]:
affected_jobs = (
    job_descriptions[
        job_descriptions["job_id"].isin(
            misranked["job_id"].unique()
        )
    ][
        [
            "job_id",
            "job_title",
            "min_experience_years",
            "required_skills",
            "nice_to_have_skills",
            "description_text"
        ]
    ]
)

In [252]:
for _, row in affected_jobs.iterrows():

    print("\n" + "-" * 75)
    print("JOB:", row["job_id"], "-", row["job_title"])
    print("Minimum experience:", row["min_experience_years"])
    print("Required skills:", row["required_skills"])
    print("Preferred skills:", row["nice_to_have_skills"])
    print("Description:", row["description_text"])


---------------------------------------------------------------------------
JOB: J02 - Senior Software Engineer
Minimum experience: 6
Required skills: Python; System Design; REST APIs; PostgreSQL; Docker
Preferred skills: Kubernetes; AWS; Microservices; CI/CD; Mentoring
Description: Zenith Nova Technologies is hiring a Senior Software Engineer for the D01 team. You will own day-to-day delivery in your area, work closely with colleagues across departments and help us improve how we serve customers. Must have: Python, System Design, REST APIs, PostgreSQL, Docker. Good to have: Kubernetes, AWS, Microservices, CI/CD. Minimum 6 year(s) of relevant experience. Education: B.E./B.Tech or MCA. Location: Bengaluru, Chennai or Hyderabad (hybrid).

---------------------------------------------------------------------------
JOB: J04 - ML Engineer
Minimum experience: 3
Required skills: Python; Machine Learning; scikit-learn; SQL; Model Deployment
Preferred skills: TensorFlow; PyTorch; MLOps; Docker

In [253]:
job_skill_audit = []

for _, job in job_descriptions.iterrows():

    description = str(job["description_text"])

    required_skills = parse_skills(job["required_skills"])
    preferred_skills = parse_skills(job["nice_to_have_skills"])

    missing_required = []
    missing_preferred = []

    for skill in required_skills:
        pattern = r"(?<!\w)" + re.escape(skill) + r"(?!\w)"

        if not re.search(pattern, description, flags=re.IGNORECASE):
            missing_required.append(skill)

    for skill in preferred_skills:
        pattern = r"(?<!\w)" + re.escape(skill) + r"(?!\w)"

        if not re.search(pattern, description, flags=re.IGNORECASE):
            missing_preferred.append(skill)

    job_skill_audit.append({
        "job_id": job["job_id"],
        "job_title": job["job_title"],
        "missing_required_skills": missing_required,
        "missing_preferred_skills": missing_preferred
    })

job_skill_audit = pd.DataFrame(job_skill_audit)

display(job_skill_audit)

,job_id,job_title,missing_required_skills,missing_preferred_skills
0,J01,Software Engineer,[],"[unit testing, postgresql]"
1,J02,Senior Software Engineer,[],[mentoring]
2,J03,Data Analyst,[],[apache superset]
3,J04,ML Engineer,[],[nlp]
4,J05,Product Manager,[],[]
5,J06,Customer Support Executive,[],[]
6,J07,HR Executive,[],[]
7,J08,Accountant,[],[]
8,J09,Account Executive,[],[]
9,J10,Digital Marketing Executive,[],[]


In [259]:
total_hits = ranking_quality["model_top5_hits"].sum()
total_shortlisted = ranking_quality["human_shortlisted"].sum()
total_top5_positions = len(ranking_quality) * 5
print("OVERALL UPDATED RANKING QUALITY")
print("Precision@5:", round(total_hits / total_top5_positions * 100, 2), "%")
print("Recall@5:", round(total_hits / total_shortlisted * 100, 2), "%")
print("Mean NDCG@5:", round(ranking_quality["ndcg_at_5"].mean(), 4))
print("Mean top-5 overlap:", round(ranking_quality["top5_overlap"].mean(), 2))

OVERALL UPDATED RANKING QUALITY
Precision@5: 94.0 %
Recall@5: 49.47 %
Mean NDCG@5: 0.9839
Mean top-5 overlap: 4.6


In [275]:
baseline_comparison = ranking_results.copy()
resume_positions = {
    resume_id: i
    for i, resume_id in enumerate(resumes["resume_id"])
}
job_positions = {
    job_id: j
    for j, job_id in enumerate(job_descriptions["job_id"])}
baseline_comparison["old_tfidf_similarity"] = [ old_similarity_check[resume_positions[resume_id],job_positions[job_id] ] for resume_id, job_id in zip( baseline_comparison["resume_id"],  baseline_comparison["job_id"] )]
baseline_comparison["old_ranking_score"] = 100 * ( 0.40 * baseline_comparison["old_tfidf_similarity"] + 0.35 * baseline_comparison["required_skill_coverage"]+ 0.15 * baseline_comparison["preferred_skill_coverage"]+ 0.10 * baseline_comparison["experience_relevance"])
old_ranked = baseline_comparison.sort_values(["job_id", "old_ranking_score"],ascending=[True, False], kind="mergesort").copy()

In [276]:
old_ranked["old_model_rank"] =  (old_ranked.groupby("job_id").cumcount() + 1)
new_ranked = ranking_results.sort_values(["job_id", "ranking_score"],ascending=[True, False],kind="mergesort").copy()

In [277]:
new_ranked["new_model_rank"] = (
    new_ranked.groupby("job_id").cumcount() + 1
)
rank_comparison = old_ranked[
    ["resume_id", "job_id", "old_ranking_score", "old_model_rank"]
].merge(
    new_ranked[
        ["resume_id", "job_id", "ranking_score", "new_model_rank"]
    ],
    on=["resume_id", "job_id"],
    how="inner"
)
rank_comparison["rank_changed"] = (
    rank_comparison["old_model_rank"]
    != rank_comparison["new_model_rank"]
)

rank_comparison["top5_changed"] = (
    (rank_comparison["old_model_rank"] <= 5)
    != (rank_comparison["new_model_rank"] <= 5)
)

In [280]:
print("Total comparisons:", len(rank_comparison))
print("Candidates with changed rank:",rank_comparison["rank_changed"].sum())
print("Candidates with changed top-5 membership:",rank_comparison["top5_changed"].sum())
print("Average absolute rank movement:",round((rank_comparison["old_model_rank"]- rank_comparison["new_model_rank"]).abs().mean(),3))

Total comparisons: 1850
Candidates with changed rank: 64
Candidates with changed top-5 membership: 0
Average absolute rank movement: 0.044


In [281]:
display(rank_comparison[rank_comparison["rank_changed"]].sort_values(["job_id", "new_model_rank"]).head(20))

,resume_id,job_id,old_ranking_score,old_model_rank,ranking_score,new_model_rank,rank_changed,top5_changed
140,RES-0029,J01,10.196422,141,10.197873,139,True,False
138,RES-0060,J01,10.197304,139,10.197433,140,True,False
139,RES-0157,J01,10.196602,140,10.196663,141,True,False
957,RES-0147,J06,18.028195,33,18.039062,31,True,False
955,RES-0110,J06,18.030384,31,18.030384,32,True,False
956,RES-0170,J06,18.030384,32,18.030384,33,True,False
1005,RES-0081,J06,11.052878,81,11.057824,80,True,False
1004,RES-0013,J06,11.057082,80,11.057082,81,True,False
1025,RES-0131,J06,10.528659,101,10.531160,100,True,False
1024,RES-0050,J06,10.529333,100,10.529667,101,True,False


In [282]:
human_labels = resume_job_labels[
    [
        "resume_id",
        "job_id",
        "human_suitability_score",
        "shortlist",
        "reviewer_note"
    ]
].copy()

evaluation = ranking_results.merge(
    human_labels,
    on=["resume_id", "job_id"],
    how="inner"
)

evaluation["shortlist_flag"] = (
    evaluation["shortlist"]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("Y")
)

# Model rank among human-reviewed candidates
evaluation["model_eval_rank"] = (
    evaluation.groupby("job_id")["ranking_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Human rank based on suitability score
evaluation["human_rank"] = (
    evaluation.groupby("job_id")["human_suitability_score"]
    .rank(method="min", ascending=False)
    .astype(int)
)

# Identify false positives and false negatives
false_positives = evaluation[
    (evaluation["model_eval_rank"] <= 5)
    & (~evaluation["shortlist_flag"])
].copy()

false_negatives = evaluation[
    (evaluation["model_eval_rank"] > 5)
    & (evaluation["shortlist_flag"])
].copy()

print("Updated false positives:", len(false_positives))
print("Updated false negatives:", len(false_negatives))

# Select three false positives
selected_fp = false_positives.sort_values(
    "ranking_score",
    ascending=False
).head(3)

# Select two false negatives with highest human suitability
selected_fn = false_negatives.sort_values(
    ["human_suitability_score", "ranking_score"],
    ascending=[False, True]
).head(2)

# Combine five examples
misranked_5 = pd.concat(
    [selected_fp, selected_fn],
    ignore_index=True
)

misranked_5["error_type"] = (
    ["False Positive"] * len(selected_fp)
    + ["False Negative"] * len(selected_fn)
)

display(
    misranked_5[
        [
            "resume_id",
            "job_id",
            "error_type",
            "model_eval_rank",
            "human_rank",
            "ranking_score",
            "human_suitability_score",
            "shortlist",
            "required_skill_coverage",
            "preferred_skill_coverage",
            "experience_relevance",
            "reviewer_note"
        ]
    ]
)

Updated false positives: 3
Updated false negatives: 48


,resume_id,job_id,error_type,model_eval_rank,human_rank,ranking_score,human_suitability_score,shortlist,required_skill_coverage,preferred_skill_coverage,experience_relevance,reviewer_note
0,RES-0129,J02,False Positive,3,4,46.829161,3,N,0.6,0.80,1.000000,Partial match
1,RES-0119,J02,False Positive,4,4,40.894570,3,N,0.6,0.20,0.983333,Partial match
2,RES-0077,J02,False Positive,5,4,25.806217,3,N,0.4,0.00,0.950000,Partial match
3,RES-0012,J09,False Negative,19,1,29.957975,5,Y,0.4,0.25,1.000000,Strong match
4,RES-0088,J04,False Negative,7,1,54.486866,5,Y,1.0,0.40,0.933333,Strong match


In [283]:
api_root = Path("08_api")
print("API DIRECTORY EXISTS:", api_root.exists())
print("\nAPI PROJECT FILES:")
for path in sorted(api_root.rglob("*")):
    if path.is_file() and "__pycache__" not in str(path):
        print(path)
main_file = api_root / "app" / "main.py"
if main_file.exists():
    print("\n" + "=" * 70)
    print("EXISTING MAIN.PY")
    print("=" * 70)
    print(main_file.read_text(encoding="utf-8"))
else:
    print("\nmain.py was not found at the expected location.")
models_dir = Path("models")
print("\nEXISTING MODEL ARTIFACTS:")
if models_dir.exists():
    for path in sorted(models_dir.iterdir()):
        if path.is_file():
            print(path.name)

API DIRECTORY EXISTS: True

API PROJECT FILES:
08_api/.ipynb_checkpoints/README-checkpoint.md
08_api/README.md
08_api/app/.ipynb_checkpoints/__init__-checkpoint.py
08_api/app/.ipynb_checkpoints/chatbot-checkpoint.py
08_api/app/.ipynb_checkpoints/main-checkpoint.py
08_api/app/.ipynb_checkpoints/schemas-checkpoint.py
08_api/app/__init__.py
08_api/app/chatbot.py
08_api/app/main.py
08_api/app/schemas.py
08_api/task9_backup/chatbot.py
08_api/task9_backup/main.py
08_api/task9_backup/schemas.py

EXISTING MAIN.PY
from contextlib import asynccontextmanager
from pathlib import Path
from datetime import datetime
from uuid import uuid4

import joblib
import pandas as pd
import numpy as np

from fastapi import FastAPI, Request, HTTPException
from .schemas import (
    AttendanceRequest,
    PayrollRequest,
    AnomalyResponse,
    PayrollBatchRequest,
    ChatRequest,
    ChatResponse
)
from .chatbot import CortexHRChatbot

PROJECT_ROOT = Path(__file__).resolve().parents[2]
MODELS_DIR = PROJECT_ROO

In [285]:
schema_path = Path("08_api/app/schemas.py")

task11_dir = Path("reports/task11")
if task11_dir.exists():
    for path in sorted(task11_dir.rglob("*")):
        if path.is_file():
            print(path)
else:
    print("reports/task11 directory not found")
print("\n" + "=" * 60)
print("TASK 11 NOTEBOOK VARIABLES")
print("=" * 60)
for variable in [
    "resumes",
    "job_descriptions",
    "ranking_results",
    "ranked_candidates",
    "ranking_quality"
]:
    print(variable, ":", variable in globals())
print("\n" + "=" * 60)
print("MODEL DIRECTORY")
print("=" * 60)
for path in sorted(Path("models").glob("*")):
    print(path.name)

reports/task11/.ipynb_checkpoints/Untitled-checkpoint.ipynb
reports/task11/.ipynb_checkpoints/all_resume_job_rankings-checkpoint.csv
reports/task11/.ipynb_checkpoints/misranked_candidates-checkpoint.csv
reports/task11/.ipynb_checkpoints/ranking_quality-checkpoint.csv
reports/task11/Untitled.ipynb
reports/task11/all_resume_job_rankings.csv
reports/task11/final_all_resume_job_scores.csv
reports/task11/final_ranked_candidates.csv
reports/task11/final_ranking_quality.csv
reports/task11/final_tfidf_similarity.csv
reports/task11/misranked_candidates.csv
reports/task11/ranking_quality.csv

TASK 11 NOTEBOOK VARIABLES
resumes : True
job_descriptions : True
ranking_results : True
ranked_candidates : True
ranking_quality : True

MODEL DIRECTORY
.ipynb_checkpoints
attendance_detector.joblib
attrition_preprocessor.joblib
chatbot_bundle.joblib
metadata.json
payroll_detector.joblib
reason_codes.joblib


In [286]:
required_columns = [
    "resume_id",
    "job_id",
    "tfidf_similarity",
    "required_skill_coverage",
    "preferred_skill_coverage",
    "experience_relevance",
    "ranking_score"
]

In [287]:
assert all(
    col in ranking_results.columns
    for col in required_columns
), "Missing ranking columns"
assert len(ranking_results) == 1850
assert ranking_results["resume_id"].nunique() == 185
assert ranking_results["job_id"].nunique() == 10
assert not ranking_results.duplicated(["resume_id", "job_id"]).any()

In [288]:
candidate_profiles = resumes[
    [
        "resume_id",
        "current_title",
        "extracted_experience_years",
        "extracted_skills"
    ]
].copy()


In [289]:
candidate_profiles["extracted_skills"] = (candidate_profiles["extracted_skills"].apply(lambda skills: sorted(set(skills))))
candidate_profiles = candidate_profiles.rename(columns={"current_title": "candidate_title","extracted_experience_years": "experience_years"})

In [294]:
def convert_skills(value):
    if pd.isna(value):
        return []

    return sorted({
        skill.strip().lower()
        for skill in str(value).split(";")
        if skill.strip()
    })

In [295]:
job_catalog = {}
for _, job in job_descriptions.iterrows():

    job_catalog[str(job["job_id"])] = {
        "job_title": str(job["job_title"]),
        "minimum_experience_years": float(
            job["min_experience_years"]
        ),
        "required_skills": convert_skills(
            job["required_skills"]
        ),
        "preferred_skills": convert_skills(
            job["nice_to_have_skills"]
        )
    }

In [296]:
ranking_table = ranking_results[required_columns].copy()
ranking_table["resume_id"] = (ranking_table["resume_id"].astype(str))
ranking_table["job_id"] = (ranking_table["job_id"].astype(str))

In [297]:
resume_ranking_bundle = {
    "model_version": "resume-ranker-1.0",
    "scoring_weights": {
        "tfidf_similarity": 0.40,
        "required_skill_coverage": 0.35,
        "preferred_skill_coverage": 0.15,
        "experience_relevance": 0.10
    },"candidates": candidate_profiles,"jobs": job_catalog,"rankings": ranking_table}

In [298]:
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)
artifact_path = (models_dir / "resume_ranking_bundle.joblib")
joblib.dump(resume_ranking_bundle,artifact_path,compress=3)

['models/resume_ranking_bundle.joblib']

In [299]:
loaded_bundle = joblib.load(artifact_path)

In [300]:
print("RESUME RANKING ARTIFACT CREATED")
print("--------------------------------")
print("Artifact:", artifact_path)
print("File size:", round(
    artifact_path.stat().st_size / 1024, 2
), "KB")
print("Candidates:", len(loaded_bundle["candidates"]))
print("Jobs:", len(loaded_bundle["jobs"]))
print("Ranking records:", len(loaded_bundle["rankings"]))
print("Model version:", loaded_bundle["model_version"])
print("Bundle keys:", list(loaded_bundle.keys()))

RESUME RANKING ARTIFACT CREATED
--------------------------------
Artifact: models/resume_ranking_bundle.joblib
File size: 35.08 KB
Candidates: 185
Jobs: 10
Ranking records: 1850
Model version: resume-ranker-1.0
Bundle keys: ['model_version', 'scoring_weights', 'candidates', 'jobs', 'rankings']


In [ ]:
#resume ranking schemas

In [301]:
schema_path = Path("08_api/app/schemas.py")

In [302]:
schema_text = schema_path.read_text(encoding="utf-8")

In [303]:
if "class ResumeRankRequest" not in schema_text:

    new_schemas = '''

# ============================================================
# TASK 11: RESUME RANKING API SCHEMAS
# ============================================================

class ResumeRankRequest(BaseModel):

    model_config = ConfigDict(extra="forbid")

    job_id: str = Field(
        min_length=1,
        max_length=20
    )

    top_k: int = Field(
        default=5,
        ge=1,
        le=20
    )


class ResumeRankCandidate(BaseModel):

    model_config = ConfigDict(extra="forbid")

    rank: int = Field(ge=1)

    resume_id: str

    candidate_title: str

    experience_years: float = Field(ge=0)

    extracted_skills: list[str]

    ranking_score: float = Field(ge=0, le=100)

    tfidf_similarity: float = Field(ge=0, le=1)

    required_skill_coverage: float = Field(ge=0, le=1)

    preferred_skill_coverage: float = Field(ge=0, le=1)

    experience_relevance: float = Field(ge=0, le=1)

    matched_required_skills: list[str]

    missing_required_skills: list[str]

    matched_preferred_skills: list[str]

    explanation: list[str]


class ResumeRankResponse(BaseModel):

    model_config = ConfigDict(extra="forbid")

    job_id: str

    job_title: str

    model_version: str

    total_candidates: int = Field(ge=0)

    returned_candidates: int = Field(ge=0)

    scoring_weights: dict[str, float]

    candidates: list[ResumeRankCandidate]

    score_interpretation: str
'''

    schema_path.write_text(
        schema_text.rstrip() + "\n" + new_schemas,
        encoding="utf-8"
    )

    print("Resume ranking schemas added successfully.")

else:
    print("Resume ranking schemas already exist. No duplicate added.")


# Verify the new definitions
updated_schemas = schema_path.read_text(encoding="utf-8")

for class_name in [
    "ResumeRankRequest",
    "ResumeRankCandidate",
    "ResumeRankResponse"
]:
    print(
        class_name,
        ":", "FOUND" if f"class {class_name}" in updated_schemas else "MISSING"
    )

Resume ranking schemas added successfully.
ResumeRankRequest : FOUND
ResumeRankCandidate : FOUND
ResumeRankResponse : FOUND


In [304]:
main_path = Path("08_api/app/main.py")
main_text = main_path.read_text(encoding="utf-8")

In [306]:
for name in [
    "ResumeRankRequest",
    "ResumeRankCandidate",
    "ResumeRankResponse"
]:
    assert f"class {name}" in schema_text, f"{name} missing"
def replace_once(text, old, new, label):
    count = text.count(old)
    if count != 1:
        raise ValueError(
            f"{label}: expected one matching block, found {count}"
        )
    return text.replace(old, new, 1)
old_import = """    ChatRequest,
    ChatResponse
)"""
new_import = """    ChatRequest,
    ChatResponse,
    ResumeRankRequest,
    ResumeRankResponse
)"""
if "ResumeRankRequest" not in main_text:
    main_text = replace_once(
        main_text,
        old_import,
        new_import,
        "Schema imports"
    )
if "resume_ranking_path" not in main_text:
    main_text = replace_once(
        main_text,
        '    chatbot_path = MODELS_DIR / "chatbot_bundle.joblib"',
        '    chatbot_path = MODELS_DIR / "chatbot_bundle.joblib"\n'
        '    resume_ranking_path = MODELS_DIR / "resume_ranking_bundle.joblib"',
        "Artifact path"
    )
if "resume_ranking_path," not in main_text:
    main_text = replace_once(
        main_text,
        """        reason_codes_path,
        chatbot_path
    ]""",
        """        reason_codes_path,
        chatbot_path,
        resume_ranking_path
    ]""",
        "Required artifacts"
    )

if '"resume_ranking":' not in main_text:

    main_text = replace_once(
        main_text,
        """        "reason_codes": joblib.load(reason_codes_path)
    }""",
        """        "reason_codes": joblib.load(reason_codes_path),
        "resume_ranking": joblib.load(resume_ranking_path)
    }""",
        "Artifact loading"
    )

# add endpoint

endpoint_code = '''

# ============================================================
# TASK 11: RESUME RANKING ENDPOINT
# ============================================================

@app.post(
    "/v1/resumes/rank",
    response_model=ResumeRankResponse
)
def rank_resumes(
    payload: ResumeRankRequest,
    request: Request
):

    bundle = request.app.state.artifacts["resume_ranking"]

    job_id = payload.job_id

    # Validate job ID
    if job_id not in bundle["jobs"]:
        raise HTTPException(
            status_code=404,
            detail=f"Job ID not found: {job_id}"
        )

    job = bundle["jobs"][job_id]

    # Retrieve ranking records for the requested job
    job_rankings = bundle["rankings"][
        bundle["rankings"]["job_id"] == job_id
    ].copy()

    if job_rankings.empty:
        raise HTTPException(
            status_code=404,
            detail=f"No candidate rankings available for {job_id}"
        )

    # Sort candidates by ranking score
    job_rankings = job_rankings.sort_values(
        "ranking_score",
        ascending=False,
        kind="mergesort"
    ).reset_index(drop=True)

    # Prepare candidate profile lookup
    candidate_profiles = bundle["candidates"].set_index(
        "resume_id"
    )

    required_skills = set(job["required_skills"])
    preferred_skills = set(job["preferred_skills"])

    candidate_responses = []

    # Return the requested number of candidates
    for rank, (_, row) in enumerate(
        job_rankings.head(payload.top_k).iterrows(),
        start=1
    ):

        resume_id = row["resume_id"]

        if resume_id not in candidate_profiles.index:
            continue

        profile = candidate_profiles.loc[resume_id]

        candidate_skills = set(
            profile["extracted_skills"]
        )

        matched_required = sorted(
            candidate_skills.intersection(required_skills)
        )

        missing_required = sorted(
            required_skills.difference(candidate_skills)
        )

        matched_preferred = sorted(
            candidate_skills.intersection(preferred_skills)
        )

        # Explain the scoring components
        explanations = [
            (
                f"Required skills matched: "
                f"{len(matched_required)} of {len(required_skills)}."
            ),
            (
                f"Preferred skills matched: "
                f"{len(matched_preferred)} of {len(preferred_skills)}."
            ),
            (
                f"Candidate experience: "
                f"{float(profile['experience_years']):.1f} years; "
                f"job minimum: {job['minimum_experience_years']:.1f} years."
            ),
            (
                f"Text similarity contributed "
                f"{float(row['tfidf_similarity']) * 100:.2f}% similarity."
            )
        ]

        candidate_responses.append({

            "rank": rank,

            "resume_id": str(resume_id),

            "candidate_title": str(
                profile["candidate_title"]
            ),

            "experience_years": float(
                profile["experience_years"]
            ),

            "extracted_skills": sorted(candidate_skills),

            "ranking_score": float(
                row["ranking_score"]
            ),

            "tfidf_similarity": float(
                row["tfidf_similarity"]
            ),

            "required_skill_coverage": float(
                row["required_skill_coverage"]
            ),

            "preferred_skill_coverage": float(
                row["preferred_skill_coverage"]
            ),

            "experience_relevance": float(
                row["experience_relevance"]
            ),

            "matched_required_skills": matched_required,

            "missing_required_skills": missing_required,

            "matched_preferred_skills": matched_preferred,

            "explanation": explanations
        })


    return {

        "job_id": job_id,

        "job_title": job["job_title"],

        "model_version": bundle["model_version"],

        "total_candidates": len(job_rankings),

        "returned_candidates": len(candidate_responses),

        "scoring_weights": bundle["scoring_weights"],

        "candidates": candidate_responses,

        "score_interpretation": (
            "Ranking score is a weighted matching score out of 100, "
            "not a probability of hiring. It supports recruiter review "
            "and must not replace human judgement."
        )
    }

'''
route_marker = '@app.post(\n    "/v1/resumes/rank"'
if route_marker not in main_text:
    main_text = main_text.rstrip() + "\n\n" + endpoint_code
ast.parse(main_text)
backup_path = Path("08_api/app/main.py.task11_backup")
if not backup_path.exists():
    shutil.copy2(main_path, backup_path)
main_path.write_text(main_text,encoding="utf-8")
print("TASK 11 API INTEGRATION SAVED")
print("--------------------------------")
print("Backup:", backup_path)
print("Updated file:", main_path)
print("Resume ranking schemas imported:", "ResumeRankRequest" in main_text)
print("Resume artifact loaded:", '"resume_ranking":' in main_text)
print("Ranking endpoint added:", route_marker in main_text)
print("Python syntax:", "VALID")

TASK 11 API INTEGRATION SAVED
--------------------------------
Backup: 08_api/app/main.py.task11_backup
Updated file: 08_api/app/main.py
Resume ranking schemas imported: True
Resume artifact loaded: True
Ranking endpoint added: True
Python syntax: VALID


In [315]:
#testing API

In [319]:
api_dir = str(Path("08_api").resolve())
if api_dir not in sys.path:
    sys.path.insert(0, api_dir)
import app.main as api_main
importlib.reload(api_main)
with TestClient(api_main.app) as client:
    health = client.get("/health")
    print("HEALTH STATUS:", health.status_code)
    print(health.json())
    response = client.post(
        "/v1/resumes/rank",
        json={
            "job_id": "J04",
            "top_k": 5}
    )
    print("\nRESUME RANKING STATUS:", response.status_code)
    print(response.json())
    invalid = client.post(
        "/v1/resumes/rank",
        json={
            "job_id": "J99",
            "top_k": 5}
    )
    print("\nINVALID JOB STATUS:", invalid.status_code)
    print(invalid.json())

Chatbot artifacts loaded successfully.
All anomaly service artifacts loaded successfully.
HEALTH STATUS: 200
{'status': 'healthy', 'loaded_artifacts': ['attendance', 'payroll', 'reason_codes', 'resume_ranking']}

RESUME RANKING STATUS: 200
{'job_id': 'J04', 'job_title': 'ML Engineer', 'model_version': 'resume-ranker-1.0', 'total_candidates': 185, 'returned_candidates': 5, 'scoring_weights': {'tfidf_similarity': 0.4, 'required_skill_coverage': 0.35, 'preferred_skill_coverage': 0.15, 'experience_relevance': 0.1}, 'candidates': [{'rank': 1, 'resume_id': 'RES-0140', 'candidate_title': 'ML Engineer', 'experience_years': 6.7, 'extracted_skills': ['communication', 'crm', 'gst', 'machine learning', 'mlops', 'model deployment', 'nlp', 'python', 'pytorch', 'scikit-learn', 'sql', 'tensorflow'], 'ranking_score': 63.08947777758487, 'tfidf_similarity': 0.1522369444396218, 'required_skill_coverage': 1.0, 'preferred_skill_coverage': 0.8, 'experience_relevance': 1.0, 'matched_required_skills': ['machin

In [320]:
evidence_dir = Path("reports/task11")
evidence_dir.mkdir(parents=True, exist_ok=True)
api_evidence = {
    "health_check": {
        "status_code": health.status_code,
        "response": health.json()
    },
    "ranking_request": {
        "method": "POST",
        "endpoint": "/v1/resumes/rank",
        "payload": {
            "job_id": "J04",
            "top_k": 5
        }
    },
    "ranking_response": {
        "status_code": response.status_code,
        "response": response.json()
    },
    "invalid_job_test": {
        "status_code": invalid.status_code,
        "response": invalid.json()
    }
}

API evidence saved: reports/task11/api_test_evidence.json
All API assertions passed.


In [ ]:
assert health.status_code == 200
assert response.status_code == 200
assert invalid.status_code == 404
assert len(response.json()["candidates"]) == 5
evidence_path = evidence_dir / "api_test_evidence.json"
with open(evidence_path, "w", encoding="utf-8") as file:
    json.dump(api_evidence, file, indent=4, ensure_ascii=False)
print("API evidence saved:", evidence_path)
print("All API assertions passed.")